# Agentの高度な使い方-ToolStrategy
## 1、ToolStrategyの複数の構造化出力方式：schemaパラメータ

In [1]:
from langchain_core.messages import HumanMessage
from dataclasses import Field, dataclass

from dotenv import load_dotenv
from langchain.agents.structured_output import ProviderStrategy
from openai import BaseModel
from scripts.regsetup import description
from traitlets.utils.descriptions import describe

from pydantic import BaseModel, Field
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy

load_dotenv(override=True)

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os
from rich import print


# Pydanticを使って構造化定義
class ContractInfo(BaseModel):
    """ユーザーの連絡先情報"""
    name: str = Field(description="ユーザーの氏名")
    email: str = Field(description="ユーザーのメールアドレス")
    phone: str = Field(description="ユーザーの電話番号")


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContractInfo),
)

response = agent.invoke({
    "messages": [
        HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはsonghk@atguigu.com、電話番号は12345678912です")
    ]
}
)
print(response)
# for msg in response["messages"]:
#     msg.pretty_print()
# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='この文章から構造化情報を抽出してください：小明さんのメールアドレスはsonghk@atguigu.com、電話番
号は12345678912です',
            additional_kwargs={},
            response_metadata={},
            id='b91945d3-79c9-481d-81d3-91d56267afc5'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 47,
                    'prompt_tokens': 118,
                    'total_tokens': 165,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.59e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.59e-05,
                        'upstream_inference_prompt_cost': 1.77e-05,
                        'upstream_inference_completions_cost': 2.82e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_4147058500',
                'id': 'gen-1785890876-LcfV1b28TSdWk8JZAKER',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fcf64-2a58-7482-a6f5-9314a2a52db9-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_UyBsgwcOk4Cm4ZF2HyzfdaaT',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 118,
                'output_tokens': 47,
                'total_tokens': 165,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'",
            name='ContractInfo',
            id='d36e8fc1-be2e-41ec-8b47-a1886fc319cc',
            tool_call_id='call_UyBsgwcOk4Cm4ZF2HyzfdaaT'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

sample2、ツール呼び出しを追加。

In [2]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print


# ツールを定義
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    顧客データベースで情報を検索する
    
    Args:
        query (str): 顧客検索文字列、例えば "田中" や "鈴木"
    
    Returns:
        str: 顧客氏名、等級、最近の購入日、累計消費額を含む顧客記録の文字列
    """
    # データベース検索結果をシミュレート
    if "田中" in query.lower():
        return "顧客記録：田中、VIP顧客、最近の購入日：2026-01-15、累計消費額：$15,000"
    elif "鈴木" in query.lower():
        return "顧客記録：鈴木、一般顧客、最近の購入日：2025-12-20、累計消費額：$3,200"
    else:
        return f"顧客{query}については記録がありません"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    感謝メールを送信する
    
    Args:
        customer (str): 顧客名、例えば "田中" や "鈴木"
        
    Returns:
        str: 送信済みの顧客名を含む確認メッセージ
    """
    return f"{customer} 様に感謝メールを送信しました"


# Pydantic Schemaを定義
class CustomerAnalysis(BaseModel):
    """顧客分析レポート"""
    customer_name: str = Field(None, description="顧客氏名")
    customer_tier: Literal["見込み顧客", "一般顧客", "VIP顧客", "離反リスク"] = Field("見込み顧客",
                                                                                  description="顧客等級。見込み顧客・一般顧客・VIP顧客・離反リスクのいずれか")
    recent_activity: str = Field(None, description="最近の活動")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消費レベル")
    send_email: bool = Field(False, description="感謝メールを送信済みかどうか")




agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "指定された顧客の状況を分析してください："
                                        "1. まず顧客データベースを検索して最新状況を把握する "
                                        "2. VIP顧客の場合は感謝メールを送信する "
                                        "3. 検索結果に基づいて構造化分析レポートを生成する "
                                        "4. ユーザーの質問が顧客記録と無関係、または顧客情報が見つからない場合は、空オブジェクトを返し、感謝メールは送信しない"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 分析を実行
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"以下のテキストの内容を抽出してください：氏名：田中，メールアドレス：zhang3@atguigu.com，イベント名：忘年会、開催日：2026-07-15"}]
})
print(result)
# for msg in result["messages"]:
# msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)


{
    'messages': [
        HumanMessage(
            content='以下のテキストの内容を抽出してください：氏名：田中，メールアドレス：zhang3@atguigu.com，イベン
ト名：忘年会、開催日：2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='c69f122d-4a47-48a3-bbd2-b58ad252d4e2'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 63,
                    'prompt_tokens': 456,
                    'total_tokens': 519,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.0001062,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.0001062,
                        'upstream_inference_prompt_cost': 6.84e-05,
                        'upstream_inference_completions_cost': 3.78e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_66ee548660',
                'id': 'gen-1785890878-aWMHzfIThCp7tYMkiCrF',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fcf64-3259-7cf0-9919-2f1cc6b6c21a-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '田中'},
                    'id': 'call_DT8p6H8BwgCs9mINLdFVMfVs',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email',
                    'args': {'customer': '田中'},
                    'id': 'call_0NB3v0yQVRQNh2h9N6d7x8ot',
                    'type': 'tool_call'
                },
                {
                    'name': 'CustomerAnalysis',
                    'args': {'customer_name': '田中'},
                    'id': 'call_4desx3WrbWWAcyBCrrPgjpr3',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 456,
                'output_tokens': 63,
                'total_tokens': 519,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: customer_name='田中' customer_tier='見込み顧客' 
recent_activity=None spending_level=None send_email=False",
            name='CustomerAnalysis',
            id='c8e12b4a-5119-4482-8728-7374459bc0a6',
            tool_call_id='call_4desx3WrbWWAcyBCrrPgjpr3'
        ),
        ToolMessage(
            content='顧客記録：田中、VIP顧客、最近の購入日：2026-01-15、累計消費額：$15,000',
            name='search_customer_database',
            id='6ebb3aa9-a718-4168-8fbb-908ce3deed6f',
            tool_call_id='call_DT8p6H8BwgCs9mINLdFVMfVs'
        ),
        ToolMessage(
            content='田中 様に感謝メールを送信しました',
            name='send_email',
            id='960d95e9-c0f8-4337-849b-c79ee31f3f15',
            tool_call_id='call_0NB3v0yQVRQNh2h9N6d7x8ot'
        )
    ],
    'structured_response': CustomerAnalysis(
        customer_name='田中',
        customer_tier='見込み顧客',
        recent_activity=None,
        spending_level=None,
        send_email=False
    )
}

出力モード2：TypedDict型

In [3]:
from langchain_core.messages import HumanMessage
from dataclasses import Field

from dotenv import load_dotenv
from langchain.agents.structured_output import ProviderStrategy
from openai import BaseModel
from scripts.regsetup import description
from traitlets.utils.descriptions import describe
from typing import TypedDict, Annotated

from pydantic import BaseModel, Field
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy

load_dotenv(override=True)

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os
from rich import print


# # Pydanticを使って構造化定義
# class ContractInfo(BaseModel):
#     """ユーザーの連絡先情報"""
#     name: str = Field(description="ユーザーの氏名")
#     email: str = Field(description="ユーザーのメールアドレス")
#     phone: str = Field(description="ユーザーの電話番号")


# Pydanticを使って構造化定義
class ContractInfo(BaseModel):
    """ユーザーの連絡先情報"""
    name: Annotated[str, ..., "ユーザー氏名"]
    email: Annotated[str, ..., "ユーザーのメールアドレス"]
    phone: Annotated[str, ..., "ユーザーの電話番号"]


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContractInfo),
)

response = agent.invoke({
    "messages": [
        HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはsonghk@atguigu.com、電話番号は12345678912です")
    ]
}
)
print(response)
# for msg in response["messages"]:
#     msg.pretty_print()
# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='この文章から構造化情報を抽出してください：小明さんのメールアドレスはsonghk@atguigu.com、電話番
号は12345678912です',
            additional_kwargs={},
            response_metadata={},
            id='72f1e284-ca7f-4dd2-856f-80f84b79c1fe'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 31,
                    'prompt_tokens': 91,
                    'total_tokens': 122,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 3.225e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 3.225e-05,
                        'upstream_inference_prompt_cost': 1.365e-05,
                        'upstream_inference_completions_cost': 1.86e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_830d456649',
                'id': 'gen-1785890880-ED6TFpZVY2msAgWOSXnS',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fcf64-387b-7303-8729-ec02097d0167-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_c6p0FRd2qlRGZ9NiayqmAyvq',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 91,
                'output_tokens': 31,
                'total_tokens': 122,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'",
            name='ContractInfo',
            id='3c37b05b-5e9f-40f6-8221-c1762d159016',
            tool_call_id='call_c6p0FRd2qlRGZ9NiayqmAyvq'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

sample2

In [4]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print


# ツールを定義
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    顧客データベースで情報を検索する
    
    Args:
        query (str): 顧客検索文字列、例えば "田中" や "鈴木"
    
    Returns:
        str: 顧客氏名、等級、最近の購入日、累計消費額を含む顧客記録の文字列
    """
    # データベース検索結果をシミュレート
    if "田中" in query.lower():
        return "顧客記録：田中、VIP顧客、最近の購入日：2026-01-15、累計消費額：$15,000"
    elif "鈴木" in query.lower():
        return "顧客記録：鈴木、一般顧客、最近の購入日：2025-12-20、累計消費額：$3,200"
    else:
        return f"顧客{query}については記録がありません"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    感謝メールを送信する
    
    Args:
        customer (str): 顧客名、例えば "田中" や "鈴木"
        
    Returns:
        str: 送信済みの顧客名を含む確認メッセージ
    """
    return f"{customer} 様に感謝メールを送信しました"


# TypedDict を使って顧客分析レポート Schema を定義
class CustomerAnalysis(TypedDict):
    """顧客分析レポート"""
    customer_name: Annotated[Optional[str], None, "顧客氏名"]
    customer_tier: Annotated[Literal["見込み顧客", "一般顧客", "VIP顧客", "離反リスク"], "見込み顧客", "顧客等級"]
    recent_activity: Annotated[Optional[str], None, "最近の活動"]
    spending_level: Annotated[Optional[Literal["低", "中", "高"]], None, "消費レベル"]
    send_email: Annotated[bool, False, "感謝メールを送信済みかどうか"]

    # エージェントを作成


agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "指定された顧客の状況を分析してください："
                                        "1. まず顧客データベースを検索して最新状況を把握する "
                                        "2. VIP顧客の場合は感謝メールを送信する "
                                        "3. 検索結果に基づいて構造化分析レポートを生成する "
                                        "4. ユーザーの質問が顧客記録と無関係、または顧客情報が見つからない場合は、空オブジェクトを返し、感謝メールは送信しない"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 分析を実行
result = agent.invoke({
    "messages": [{"role": "user", "content": "田中という顧客を分析してください"}]
    # "messages": [{"role": "user","content": "鈴木という顧客を分析してください"}]
    # "messages": [{"role": "user","content": "佐藤という顧客を分析してください"}]
    # "messages": [{"role": "user","content": "今日の天気はどうですか"}]
})

# 結果を処理
print(result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)


{
    'messages': [
        HumanMessage(
            content='田中という顧客を分析してください',
            additional_kwargs={},
            response_metadata={},
            id='988e8aa4-d749-4042-959e-33e6d1a70e77'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 16,
                    'prompt_tokens': 330,
                    'total_tokens': 346,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 5.91e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 5.91e-05,
                        'upstream_inference_prompt_cost': 4.95e-05,
                        'upstream_inference_completions_cost': 9.6e-06
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_66ee548660',
                'id': 'gen-1785890880-u7Y1bMovrDAb6vvsU8bS',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fcf64-3c2c-7271-859c-66c2b0594440-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '田中'},
                    'id': 'call_4CVepRijXVZt4CUCVkp6QYwI',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 330,
                'output_tokens': 16,
                'total_tokens': 346,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='顧客記録：田中、VIP顧客、最近の購入日：2026-01-15、累計消費額：$15,000',
            name='search_customer_database',
            id='fc4cdac8-6108-4e14-adb5-136ff2303e13',
            tool_call_id='call_4CVepRijXVZt4CUCVkp6QYwI'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 82,
                    'prompt_tokens': 391,
                    'total_tokens': 473,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00010785,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00010785,
                        'upstream_inference_prompt_cost': 5.865e-05,
                        'upstream_inference_completions_cost': 4.92e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
          

json schema

In [5]:
from langchain_core.messages import HumanMessage
from dataclasses import Field
from dataclasses import dataclass
from dotenv import load_dotenv
from langchain.agents.structured_output import ProviderStrategy
from openai import BaseModel
from scripts.regsetup import description
from traitlets.utils.descriptions import describe

from pydantic import BaseModel, Field
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy

load_dotenv(override=True)

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os
from rich import print

json_schema = {
    "title": "ContactInfo",
    "description": "ユーザーの連絡先情報",
    "type": "object",
    "properties": {
        "name": {
            "description": "ユーザー氏名",
            "type": "string"
        },
        "email": {
            "description": "ユーザーのメールアドレス",
            "type": "string"
        },
        "phone": {
            "description": "ユーザーの携帯電話番号",
            "type": "string"
        }
    },
    "required": [
        "name",
        "email",
        "phone"
    ]
}

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

agent = create_agent(
    model=model,
    response_format=ToolStrategy(json_schema),
)

response = agent.invoke({
    "messages": [
        HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはsonghk@atguigu.com、電話番号は12345678912です")
    ]
}
)
print(response)
# for msg in response["messages"]:
#     msg.pretty_print()
# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='この文章から構造化情報を抽出してください：小明さんのメールアドレスはsonghk@atguigu.com、電話番
号は12345678912です',
            additional_kwargs={},
            response_metadata={},
            id='b322db25-a19c-4fb4-98b1-04d6c720b06d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 47,
                    'prompt_tokens': 119,
                    'total_tokens': 166,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.605e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.605e-05,
                        'upstream_inference_prompt_cost': 1.785e-05,
                        'upstream_inference_completions_cost': 2.82e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_ab0a2ab924',
                'id': 'gen-1785890884-Vu2RLA3HZJdCkhklQFaf',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fcf64-4a28-7bb3-bd25-88bc62c79dbc-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_fd5EwePBYUd9iBIHChKWCy2z',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 119,
                'output_tokens': 47,
                'total_tokens': 166,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': 
'12345678912'}",
            name='ContactInfo',
            id='26338376-3520-48a5-9f22-e80c23b43cb7',
            tool_call_id='call_fd5EwePBYUd9iBIHChKWCy2z'
        )
    ],
    'structured_response': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'}
}

dataclass

In [6]:
from langchain_core.messages import HumanMessage
from dataclasses import Field
from dataclasses import dataclass
from dotenv import load_dotenv
from langchain.agents.structured_output import ProviderStrategy
from openai import BaseModel
from scripts.regsetup import description
from traitlets.utils.descriptions import describe

from pydantic import BaseModel, Field
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy

load_dotenv(override=True)

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os
from rich import print


# dataclassを使って構造化定義
@dataclass
class ContractInfo():
    """ユーザーの連絡先情報"""
    name: str = Field(description="ユーザーの氏名")
    email: str = Field(description="ユーザーのメールアドレス")
    phone: str = Field(description="ユーザーの電話番号")


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContractInfo),
)

response = agent.invoke({
    "messages": [
        HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはsonghk@atguigu.com、電話番号は12345678912です")
    ]
}
)
print(response)
# for msg in response["messages"]:
#     msg.pretty_print()
# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='この文章から構造化情報を抽出してください：小明さんのメールアドレスはsonghk@atguigu.com、電話番
号は12345678912です',
            additional_kwargs={},
            response_metadata={},
            id='b9c97517-90be-40e1-bb39-f538ffa92538'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 47,
                    'prompt_tokens': 118,
                    'total_tokens': 165,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.59e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.59e-05,
                        'upstream_inference_prompt_cost': 1.77e-05,
                        'upstream_inference_completions_cost': 2.82e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_4147058500',
                'id': 'gen-1785890885-8qpTSjfowWHtcw3kvNvf',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fcf64-4e2a-7a33-834c-ae274688f767-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_bwTfwUs9pZCyg6HrJjIdsGcR',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 118,
                'output_tokens': 47,
                'total_tokens': 165,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: ContractInfo(name='小明', email='songhk@atguigu.com', 
phone='12345678912')",
            name='ContractInfo',
            id='b2f471e4-af2d-470e-8f64-6459d9297cd2',
            tool_call_id='call_bwTfwUs9pZCyg6HrJjIdsGcR'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

schema

In [7]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage


class ContactInfo(BaseModel):
    """ユーザーの連絡先情報"""
    name: str = Field(description="ユーザー氏名")
    email: str = Field(description="ユーザーのメールアドレス")
    phone: str = Field(description="ユーザーの携帯電話番号")


class EventInfo(BaseModel):
    """イベント詳細"""
    event_name: str = Field(description="イベント名")
    date: str = Field(description="イベント発生日")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventInfo]
    )
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはshkstart@atguigu.com、電話番号は12345678912です")
        ]
    }
)
for msg in response["messages"]:
    msg.pretty_print()
    print(response["structured_response"])

================================ Human Message =================================

この文章から構造化情報を抽出してください：小明さんのメールアドレスはshkstart@atguigu.com、電話番号は12345678912です


ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')

================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_khD5HGOnqDejhDvMgtv07YnX)
 Call ID: call_khD5HGOnqDejhDvMgtv07YnX
  Args:
    name: 小明
    email: shkstart@atguigu.com
    phone: 12345678912


ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')

================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'


ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')

In [8]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage("この文章から構造化情報を抽出してください：2026年の大学入試共通テストの出願者数が1200万人を突破しました")
        ]
    }
)
for msg in response["messages"]:
    msg.pretty_print()
    print(response["structured_response"])


================================ Human Message =================================

この文章から構造化情報を抽出してください：2026年の大学入試共通テストの出願者数が1200万人を突破しました


EventInfo(event_name='大学入試共通テストの出願者数', date='2026年')

================================== Ai Message ==================================
Tool Calls:
  EventInfo (call_qSdpRHbCVTgZA19XvBYGErNS)
 Call ID: call_qSdpRHbCVTgZA19XvBYGErNS
  Args:
    event_name: 大学入試共通テストの出願者数
    date: 2026年


EventInfo(event_name='大学入試共通テストの出願者数', date='2026年')

================================= Tool Message =================================
Name: EventInfo

Returning structured response: event_name='大学入試共通テストの出願者数' date='2026年'


EventInfo(event_name='大学入試共通テストの出願者数', date='2026年')

tool_message_content

In [9]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage


class ContactInfo(BaseModel):
    """ユーザーの連絡先情報"""
    name: str = Field(description="ユーザー氏名")
    email: str = Field(description="ユーザーのメールアドレス")
    phone: str = Field(description="ユーザーの携帯電話番号")


class EventInfo(BaseModel):
    """イベント詳細"""
    event_name: str = Field(description="イベント名")
    date: str = Field(description="イベント発生日")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema=ContactInfo,
        tool_message_content="情報の抽出に成功しました"
    )
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("この文章から構造化情報を抽出してください：小明さんのメールアドレスはshkstart@atguigu.com、電話番号は12345678912です")
        ]
    }
)

print(response)
# for msg in response["messages"]:
#     msg.pretty_print()
#     print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='この文章から構造化情報を抽出してください：小明さんのメールアドレスはshkstart@atguigu.com、電話
番号は12345678912です',
            additional_kwargs={},
            response_metadata={},
            id='6b45e3b3-a912-4033-8852-dae940d09e1e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 48,
                    'prompt_tokens': 120,
                    'total_tokens': 168,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 4.68e-05,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 4.68e-05,
                        'upstream_inference_prompt_cost': 1.8e-05,
                        'upstream_inference_completions_cost': 2.88e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-4o-mini',
                'system_fingerprint': 'fp_ab0a2ab924',
                'id': 'gen-1785890888-AhoVtk4Di3c0iC34CY3B',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fcf64-5b38-76a2-908e-3143e89c3019-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_Kjy2MnD4oFLyIbTNayf6ECXf',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 120,
                'output_tokens': 48,
                'total_tokens': 168,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='情報の抽出に成功しました',
            name='ContactInfo',
            id='29379e99-fb5c-453b-b462-4ade1d756d71',
            tool_call_id='call_Kjy2MnD4oFLyIbTNayf6ECXf'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}